# Lab 5C: CloudTrail Logs Monitoring

In this lab, you'll **establish governance and compliance controls** by monitoring Amazon SageMaker API activities through AWS CloudTrail. You'll audit who created, modified, or deleted ML resources, build compliance reports, set up **real-time alerting** on critical governance events with EventBridge, and run automated compliance checks.

## Prerequisites

- Completed **Lab 5A** and **Lab 5B**
- SageMaker activity from earlier labs (training jobs, endpoints, model registrations) — this *is* your audit data

::: **CloudTrail is always on**: management events for the last 90 days are automatically recorded in **Event history** — nothing to enable.

## What you'll do

1. Understand CloudTrail events for SageMaker
2. Query your account's SageMaker API history
3. Build a **who/what/when audit table** and compliance reports
4. Create an **EventBridge rule** that alerts on endpoint deletion
5. Run **automated compliance checks** (approved models, tagging)
6. Learn the long-term setup: trails, Athena, security integrations (console)

> ℹ️ **Permissions note**: your notebook execution role may not include `cloudtrail:LookupEvents`. Every query cell below falls back to precise **console instructions** if the API call is denied — CloudTrail's Event history console gives you the identical data. Your Workshop Studio *console* identity has full access.


## Section 1: CloudTrail Events for SageMaker

Every SageMaker action — console, CLI, SDK, or service-to-service — produces a CloudTrail **management event** answering:

- **Who**: user identity (IAM role/user, session)
- **What**: API action (`CreateTrainingJob`, `DeleteEndpoint`, ...)
- **When / Where**: timestamp, region, source IP
- **How**: request parameters and response elements
- **Result**: success or error code

### Key SageMaker events for governance

| Event | Governance use case |
|---|---|
| `CreateTrainingJob` | Who trains models, with what data |
| `CreateModel` / `CreateModelPackage` | Model creation and registry entries |
| `UpdateModelPackage` | **Approval workflow audit** (Approved/Rejected) |
| `CreateEndpoint` / `UpdateEndpoint` | Production deployments and changes |
| `DeleteEndpoint` | Resource deletion investigations |
| Failed calls (`AccessDenied`, ...) | Unauthorized access attempts |

> `InvokeEndpoint` is a **data event** — not logged by default, and we deliberately don't enable data-event logging in this workshop (high volume/cost).


In [ ]:
import boto3
import json
import pandas as pd
from datetime import datetime, timedelta, timezone

session = boto3.session.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

ct = session.client('cloudtrail')
sm = session.client('sagemaker')
sns = session.client('sns')

print(f'Region : {region}')
print(f'Account: {account_id}')

# Probe CloudTrail access once; all later cells adapt to this flag
try:
    ct.lookup_events(MaxResults=1)
    CLOUDTRAIL_ACCESS = True
    print('✅ cloudtrail:LookupEvents available — querying from the notebook')
except Exception as e:
    CLOUDTRAIL_ACCESS = False
    print(f'⚠️ CloudTrail API not available from this role ({type(e).__name__}).')
    print('   Use the CloudTrail CONSOLE for the query sections — instructions are provided in each cell.')

In [ ]:
# 🔗 Console deep-link helpers — build clickable AWS console URLs from this notebook
import urllib.parse
from IPython.display import Markdown, display

_CW = f'https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}'
_SM = f'https://{region}.console.aws.amazon.com/sagemaker/home?region={region}'

def _star_escape(s):
    """CloudWatch graph/Insights fragment encoding: percent-encode everything, then % → * (lowercase hex)."""
    q = urllib.parse.quote(str(s), safe='')
    out, i = '', 0
    while i < len(q):
        if q[i] == '%':
            out += '*' + q[i+1:i+3].lower(); i += 3
        else:
            out += q[i]; i += 1
    return out

def _dollar_escape(s):
    """CloudWatch Logs fragment encoding: double percent-encode, then % → $."""
    return urllib.parse.quote(urllib.parse.quote(str(s), safe='')).replace('%', '$')

def url_metrics_graph(title, *metrics, stat='Average', period=60):
    """Pre-built CloudWatch metrics graph. metrics: tuples of (namespace, metric_name, dims_dict)."""
    m_parts = []
    for ns, mn, dims in metrics:
        p = [f"~'{_star_escape(ns)}", f"~'{_star_escape(mn)}"]
        for k, v in dims.items():
            p += [f"~'{_star_escape(k)}", f"~'{_star_escape(v)}"]
        m_parts.append('~(' + ''.join(p) + ')')
    graph = (f"~(metrics~({''.join(m_parts)})~view~'timeSeries~stacked~false"
             f"~region~'{region}~stat~'{_star_escape(stat)}~period~{period}~title~'{_star_escape(title)})")
    return f'{_CW}#metricsV2:graph={graph}'

def url_log_group(group, stream=None, filter_pattern=None):
    """CloudWatch Logs group (or stream) page; optional pre-set search filter."""
    u = f'{_CW}#logsV2:log-groups/log-group/{_dollar_escape(group)}'
    if stream:
        u += f'/log-events/{_dollar_escape(stream)}'
    if filter_pattern:
        u += f'$3FfilterPattern$3D{_dollar_escape(filter_pattern)}'
    return u

def url_logs_insights(query, group, hours=1):
    """Logs Insights with the query editor and log group pre-filled."""
    qd = (f"~(end~0~start~-{hours*3600}~timeType~'RELATIVE~unit~'seconds"
          f"~editorString~'{_star_escape(query)}~source~(~'{_star_escape(group)}))")
    return f'{_CW}#logsV2:logs-insights$3FqueryDetail$3D{qd}'

def url_dashboard(name):
    return f'{_CW}#dashboards/dashboard/{urllib.parse.quote(name, safe="")}'

def url_alarm(name):
    return f'{_CW}#alarmsV2:alarm/{urllib.parse.quote(name, safe="")}'

def url_sagemaker(kind, name):
    """kind: 'jobs' (training jobs) or 'endpoints'."""
    return f'{_SM}#/{kind}/{urllib.parse.quote(name, safe="")}'

def url_cloudtrail(**filters):
    """CloudTrail Event history, optionally pre-filtered (EventSource=..., EventName=...)."""
    q = urllib.parse.urlencode(filters)
    return (f'https://{region}.console.aws.amazon.com/cloudtrailv2/home?region={region}#/events'
            + (f'?{q}' if q else ''))

def url_eventbridge_rule(name):
    return (f'https://{region}.console.aws.amazon.com/events/home?region={region}'
            f'#/eventbus/default/rules/{urllib.parse.quote(name, safe="")}')

def url_sns_topic(arn):
    return f'https://{region}.console.aws.amazon.com/sns/v3/home?region={region}#/topic/{arn}'

def show_links(items):
    """Render a clickable link list. items: list of (label, url)."""
    display(Markdown('**🔗 Open in AWS Console:**\n' + '\n'.join(f'- [{l}]({u})' for l, u in items)))

print('Console deep-link helpers loaded ✅')

## Section 2: Query Your SageMaker API History

Let's pull the last 3 days of SageMaker management events and shape them into an audit table.

**Console path (always works)**: **CloudTrail → Event history** → filter **Event source** = `sagemaker.amazonaws.com`.


In [ ]:
SAGEMAKER_EVENTS_OF_INTEREST = [
    'CreateTrainingJob', 'CreateModel', 'CreateEndpointConfig', 'CreateEndpoint',
    'UpdateEndpoint', 'DeleteEndpoint', 'CreateModelPackage', 'UpdateModelPackage',
    'CreateModelPackageGroup', 'DeleteModel',
]

def lookup_sagemaker_events(days=3, max_events=500):
    """Fetch recent SageMaker management events via CloudTrail LookupEvents."""
    events, token = [], None
    start = datetime.now(timezone.utc) - timedelta(days=days)
    while len(events) < max_events:
        kwargs = dict(
            LookupAttributes=[{'AttributeKey': 'EventSource', 'AttributeValue': 'sagemaker.amazonaws.com'}],
            StartTime=start, MaxResults=50,
        )
        if token:
            kwargs['NextToken'] = token
        resp = ct.lookup_events(**kwargs)
        events.extend(resp['Events'])
        token = resp.get('NextToken')
        if not token:
            break
    return events

if CLOUDTRAIL_ACCESS:
    raw_events = lookup_sagemaker_events()
    print(f'Fetched {len(raw_events)} SageMaker events from the last 3 days')
else:
    raw_events = []
    print('Console route: CloudTrail → Event history → filter Event source = sagemaker.amazonaws.com')
    print('You will see the same events this cell would have fetched.')

In [ ]:
# Shape events into an audit DataFrame: who / what / when / from where / outcome
def to_audit_row(ev):
    detail = json.loads(ev['CloudTrailEvent'])
    ident = detail.get('userIdentity', {})
    who = ident.get('arn', '') or ident.get('principalId', '')
    who = who.split('/')[-1] if who else 'unknown'
    resource = ''
    rp = detail.get('requestParameters') or {}
    for key in ('trainingJobName', 'endpointName', 'modelName', 'modelPackageGroupName', 'modelPackageArn'):
        if key in rp:
            resource = str(rp[key]).split('/')[-1]
            break
    return {
        'time': ev['EventTime'],
        'event': ev['EventName'],
        'who': who,
        'resource': resource,
        'source_ip': detail.get('sourceIPAddress', ''),
        'error': detail.get('errorCode', ''),
    }

if raw_events:
    audit_df = pd.DataFrame([to_audit_row(e) for e in raw_events]).sort_values('time', ascending=False)
    pd.set_option('display.max_colwidth', 45)
    print('Most recent 20 SageMaker API events:')
    display(audit_df.head(20))
else:
    audit_df = pd.DataFrame()

In [ ]:
# 🔗 Same data in the console — Event history pre-filtered to SageMaker
show_links([
    ('CloudTrail Event history — all SageMaker events', url_cloudtrail(EventSource='sagemaker.amazonaws.com')),
])

In [ ]:
# Event frequency: which APIs are being called, and by whom?
if not audit_df.empty:
    print('=== Events by type ===')
    print(audit_df['event'].value_counts().head(15).to_string())
    print()
    print('=== Events by identity ===')
    print(audit_df['who'].value_counts().to_string())
    print()
    print('=== Failed calls (errorCode set) ===')
    failed = audit_df[audit_df['error'] != '']
    if failed.empty:
        print('None in this window 🎉')
    else:
        display(failed[['time', 'event', 'who', 'error']].head(10))

### Deep-dive: inspect one event record in full

A single event record is the atomic unit of your audit trail. The fields worth memorizing: `userIdentity.arn`, `eventName`, `eventTime`, `sourceIPAddress`, `requestParameters`, `errorCode`.


In [ ]:
# Show the full JSON of the most interesting recent event (prefer lifecycle events over Describe*)
if raw_events:
    priority = [e for e in raw_events if e['EventName'] in SAGEMAKER_EVENTS_OF_INTEREST]
    sample = priority[0] if priority else raw_events[0]
    detail = json.loads(sample['CloudTrailEvent'])
    print(f"Full record for: {sample['EventName']} at {sample['EventTime']}")
    print(json.dumps(detail, indent=2, default=str)[:3500])
else:
    print('Console route: click any event in Event history → View event → the same JSON record appears.')

## Section 3: Targeted Governance Queries

Three investigations you'll actually run in production. Each cell shows the API version and the equivalent console filter.


In [ ]:
# Query 1: Who deleted endpoints? (incident investigation)
# Console: Event history → filter Event name = DeleteEndpoint
if CLOUDTRAIL_ACCESS:
    resp = ct.lookup_events(
        LookupAttributes=[{'AttributeKey': 'EventName', 'AttributeValue': 'DeleteEndpoint'}],
        StartTime=datetime.now(timezone.utc) - timedelta(days=7), MaxResults=25,
    )
    if resp['Events']:
        for ev in resp['Events']:
            d = json.loads(ev['CloudTrailEvent'])
            who = d.get('userIdentity', {}).get('arn', 'unknown').split('/')[-1]
            what = (d.get('requestParameters') or {}).get('endpointName', '?')
            print(f"{ev['EventTime']}  {who:30s} deleted endpoint '{what}' from {d.get('sourceIPAddress')}")
    else:
        print('No DeleteEndpoint events in the last 7 days.')

In [ ]:
# Query 2: Model approval audit trail (UpdateModelPackage)
# Console: Event history → filter Event name = UpdateModelPackage
if CLOUDTRAIL_ACCESS:
    resp = ct.lookup_events(
        LookupAttributes=[{'AttributeKey': 'EventName', 'AttributeValue': 'UpdateModelPackage'}],
        StartTime=datetime.now(timezone.utc) - timedelta(days=30), MaxResults=25,
    )
    if resp['Events']:
        print('Model approval status changes (last 30 days):')
        for ev in resp['Events']:
            d = json.loads(ev['CloudTrailEvent'])
            rp = d.get('requestParameters') or {}
            who = d.get('userIdentity', {}).get('arn', 'unknown').split('/')[-1]
            pkg = str(rp.get('modelPackageArn', '?')).split('/')[-2:]
            status = rp.get('modelApprovalStatus', '?')
            print(f"  {ev['EventTime']}  {who:25s} → {status:25s} on {'/'.join(pkg)}")
    else:
        print('No UpdateModelPackage events in the last 30 days —')
        print('approve/reject a model version in the Model Registry (Lab 3C/4A) and re-run.')

In [ ]:
# Query 3: Training activity report — who is training what?
# Console: Event history → filter Event name = CreateTrainingJob
if CLOUDTRAIL_ACCESS:
    resp = ct.lookup_events(
        LookupAttributes=[{'AttributeKey': 'EventName', 'AttributeValue': 'CreateTrainingJob'}],
        StartTime=datetime.now(timezone.utc) - timedelta(days=30), MaxResults=50,
    )
    rows = []
    for ev in resp['Events']:
        d = json.loads(ev['CloudTrailEvent'])
        rp = d.get('requestParameters') or {}
        rows.append({
            'time': ev['EventTime'],
            'who': d.get('userIdentity', {}).get('arn', 'unknown').split('/')[-1],
            'job': rp.get('trainingJobName', '?'),
            'instance': (rp.get('resourceConfig') or {}).get('instanceType', '?'),
            'data': str(((rp.get('inputDataConfig') or [{}])[0].get('dataSource') or {})
                        .get('s3DataSource', {}).get('s3Uri', '?'))[:60],
        })
    if rows:
        print(f'{len(rows)} training job(s) created in the last 30 days:')
        display(pd.DataFrame(rows))
    else:
        print('No CreateTrainingJob events in the last 30 days.')

In [ ]:
# 🔗 Console shortcuts for the three governance queries above
show_links([
    ('Event history — DeleteEndpoint (who deleted endpoints?)', url_cloudtrail(EventName='DeleteEndpoint')),
    ('Event history — UpdateModelPackage (approval audit)', url_cloudtrail(EventName='UpdateModelPackage')),
    ('Event history — CreateTrainingJob (training activity)', url_cloudtrail(EventName='CreateTrainingJob')),
])

## Section 4: Real-Time Governance Alerts with EventBridge

Post-hoc queries are for audits; **EventBridge rules** give you *real-time* reaction. We'll wire the classic governance control: **notify immediately when anyone deletes a SageMaker endpoint**.

How it works: CloudTrail records `DeleteEndpoint` → EventBridge matches the pattern `AWS API Call via CloudTrail` → publishes to the `SageMaker-Alerts` SNS topic (from Lab 5A).

> ⚠️ Your notebook role can only manage EventBridge rules in **us-east-1**. If your event runs in another region, the cell prints console steps instead (2 minutes of clicking).


In [ ]:
# Create the EventBridge rule + SNS target
rule_name = 'SageMaker-Endpoint-Deletion-Alert'
event_pattern = {
    'source': ['aws.sagemaker'],
    'detail-type': ['AWS API Call via CloudTrail'],
    'detail': {
        'eventSource': ['sagemaker.amazonaws.com'],
        'eventName': ['DeleteEndpoint'],
    },
}

topic_arn = sns.create_topic(Name='SageMaker-Alerts')['TopicArn']  # idempotent, reuses Lab 5A topic

try:
    events_client = session.client('events')
    rule_arn = events_client.put_rule(
        Name=rule_name,
        EventPattern=json.dumps(event_pattern),
        State='ENABLED',
        Description='Alert when a SageMaker endpoint is deleted',
    )['RuleArn']
    events_client.put_targets(Rule=rule_name, Targets=[{'Id': 'sns-alert', 'Arn': topic_arn}])

    # SNS topics need a resource policy allowing EventBridge to publish
    policy = {
        'Version': '2012-10-17',
        'Statement': [{
            'Sid': 'AllowEventBridgePublish',
            'Effect': 'Allow',
            'Principal': {'Service': 'events.amazonaws.com'},
            'Action': 'sns:Publish',
            'Resource': topic_arn,
            'Condition': {'ArnEquals': {'aws:SourceArn': rule_arn}},
        }],
    }
    sns.set_topic_attributes(TopicArn=topic_arn, AttributeName='Policy', AttributeValue=json.dumps(policy))
    print(f'✅ EventBridge rule created: {rule_name}')
    print(f'   Target: {topic_arn}')
    print('   Any DeleteEndpoint call in this region now triggers an SNS notification within ~1-2 minutes.')
except Exception as e:
    print(f'⚠️ Could not create the rule from the notebook: {type(e).__name__}: {str(e)[:180]}')
    print()
    print('Create it in the console instead (EventBridge → Rules → Create rule):')
    print(f'  Name: {rule_name}, Event bus: default, Rule with an event pattern')
    print('  Pattern (paste as JSON):')
    print(json.dumps(event_pattern, indent=2))
    print(f'  Target: SNS topic → SageMaker-Alerts')

In [ ]:
# 🔗 Inspect the governance alert wiring in the console
show_links([
    ('EventBridge rule — SageMaker-Endpoint-Deletion-Alert', url_eventbridge_rule(rule_name)),
    ('SNS topic — SageMaker-Alerts', url_sns_topic(topic_arn)),
])

### More governance automations (patterns to take home)

The same CloudTrail→EventBridge mechanism powers richer automations — swap the SNS target for a **Lambda** target:

**Log every training job to a DynamoDB audit table** — pattern matches `CreateTrainingJob`; Lambda extracts `trainingJobName`, identity, instance type, and S3 data source into an item. Result: a searchable, permanent training inventory.

**Detect deployment of unapproved models** — pattern matches `CreateEndpoint`; Lambda walks endpoint→config→model→model package and checks `ModelApprovalStatus == 'Approved'`, alerting on violations. (We run a batch version of this check in Section 5.)

**Conditional auto-approval** — pattern matches `CreateModelPackage` filtered on a specific `userIdentity.arn`; Lambda calls `update_model_package(..., ModelApprovalStatus='Approved')`. Use sparingly — it weakens the manual gate.


## Section 5: Automated Compliance Checks

CloudTrail tells you what *happened*; compliance checks verify the *current state* is legal. Both are needed. These checks use only SageMaker APIs (no special permissions).


In [ ]:
# Compliance check 1: every InService endpoint must serve an APPROVED registered model
findings = []
endpoints = sm.list_endpoints()['Endpoints']
print(f'Checking {len(endpoints)} endpoint(s)...\n')

for ep in endpoints:
    name = ep['EndpointName']
    try:
        cfg_name = sm.describe_endpoint(EndpointName=name)['EndpointConfigName']
        cfg = sm.describe_endpoint_config(EndpointConfigName=cfg_name)
        for variant in cfg['ProductionVariants']:
            model = sm.describe_model(ModelName=variant['ModelName'])
            containers = model.get('Containers') or [model.get('PrimaryContainer', {})]
            pkg_arns = [c['ModelPackageName'] for c in containers if 'ModelPackageName' in c]
            if not pkg_arns:
                findings.append({'endpoint': name, 'issue': 'Model NOT from Model Registry (no lineage/approval)'})
            for arn in pkg_arns:
                status = sm.describe_model_package(ModelPackageName=arn).get('ModelApprovalStatus')
                if status != 'Approved':
                    findings.append({'endpoint': name, 'issue': f'Model package status={status} (not Approved)'})
    except Exception as e:
        findings.append({'endpoint': name, 'issue': f'Check failed: {type(e).__name__}'})

if findings:
    print('🚨 COMPLIANCE FINDINGS:')
    display(pd.DataFrame(findings))
    print('\nNote: the Lab 3A endpoint creates its model FROM the registered package, so it')
    print('should NOT be listed above. A finding here is an endpoint someone deployed straight')
    print('from an artifact URI, or one serving a package still in PendingManualApproval —')
    print('exactly what this control exists to catch.')
else:
    print('✅ All endpoints serve approved, registry-tracked models.')

In [ ]:
# Compliance check 2: required tags on endpoints (cost attribution / ownership)
REQUIRED_TAGS = ['Environment', 'Owner', 'CostCenter']

rows = []
for ep in sm.list_endpoints()['Endpoints']:
    tags = sm.list_tags(ResourceArn=ep['EndpointArn'])['Tags']
    present = {t['Key'] for t in tags}
    missing = [t for t in REQUIRED_TAGS if t not in present]
    rows.append({'endpoint': ep['EndpointName'],
                 'tags_present': sorted(present) or '—',
                 'missing_required': missing or '✅ none'})

if rows:
    display(pd.DataFrame(rows))
    print('In production: schedule this check as a Lambda (daily EventBridge schedule) and')
    print('publish findings to the SageMaker-Alerts SNS topic.')
else:
    print('No endpoints to check.')

## Section 6: Beyond 90 Days — Trails, Athena, and Security Integrations

Event history covers only **90 days**. For real governance programs, set these up from the **console** (admin task, not part of the notebook — and per workshop guardrails we don't create trails in this event account):

### 6.1 Create a trail (long-term storage)

**CloudTrail → Trails → Create trail**: name `SageMaker-Governance-Trail`, new S3 bucket, SSE-KMS encryption **on**, log file **validation on** (tamper evidence), management events **Read + Write**. Skip data events (volume/cost).

### 6.2 Query years of history with Athena

Once logs land in S3, one `CREATE EXTERNAL TABLE ... ROW FORMAT SERDE 'com.amazon.emr.hive.serde.CloudTrailSerde'` statement (CloudTrail console → Event history → **Create Athena table** generates it for you) unlocks SQL like:

```sql
-- monthly activity summary for compliance reports
SELECT eventname, count(*) AS total,
       count(DISTINCT useridentity.arn) AS unique_identities
FROM cloudtrail_logs
WHERE eventsource = 'sagemaker.amazonaws.com'
GROUP BY eventname ORDER BY total DESC;

-- after-hours activity (potential anomaly)
SELECT eventtime, useridentity.arn, eventname, sourceipaddress
FROM cloudtrail_logs
WHERE eventsource = 'sagemaker.amazonaws.com'
  AND CAST(date_format(from_iso8601_timestamp(eventtime), '%H') AS INTEGER) NOT BETWEEN 8 AND 18;
```

Alternative: **CloudTrail Lake** — managed, SQL-queryable, multi-year retention without the S3+Athena plumbing.

### 6.3 Security service integrations

| Service | What it adds on top of CloudTrail |
|---|---|
| **GuardDuty** | ML-based threat detection over CloudTrail (compromised credentials, recon, unusual API patterns) |
| **AWS Config** | Continuous resource-state compliance (e.g. `sagemaker-notebook-no-direct-internet-access`) |
| **Security Hub** | Aggregated findings + compliance standards dashboards |

### 6.4 Cost controls for CloudTrail logs

S3 lifecycle: Standard → Standard-IA (90d, $0.0125/GB) → Glacier (180d, $0.004/GB); log **data events** only for critical buckets, if ever.


## Section 7: Incident Investigation Playbook

When something unexpected happens (e.g. a production endpoint disappears), CloudTrail is your forensic record. The method:

1. **Find the event** — filter Event history by event name (`DeleteEndpoint`) + resource name → gives you *who, when, from which IP, via which tool* (user agent).
2. **Reconstruct the actor's session** — filter by that identity ± 1 hour → look for reconnaissance (`Describe*` bursts), `AccessDenied` errors (probing), privilege changes (`AttachRolePolicy`).
3. **Validate the source** — is the IP a known office/VPN? Is the time of day plausible?
4. **Build the timeline** — ordered event list = the incident report backbone.
5. **Remediate** — revoke credentials if malicious, redeploy from the Model Registry (this is why registry + approval flow matters!), then add preventive controls: the Section 4 EventBridge alert, IAM `Deny` on `DeleteEndpoint` for non-admins, MFA conditions.

Try steps 1–2 yourself now in the console using your own identity and the `DeleteEndpoint` events from earlier labs.

## Section 8: (Optional) Cleanup


In [ ]:
# Uncomment to remove the Lab 5C EventBridge rule (keep SNS topic if still using Lab 5A/5B alarms)

# events_client = session.client('events')
# events_client.remove_targets(Rule='SageMaker-Endpoint-Deletion-Alert', Ids=['sns-alert'])
# events_client.delete_rule(Name='SageMaker-Endpoint-Deletion-Alert')
# print('EventBridge rule deleted')

## Key Takeaways

✅ **CloudTrail records every SageMaker management API call** automatically — 90 days in Event history, no setup.
✅ Each event answers **who / what / when / where / how / outcome** — the atomic unit of ML governance.
✅ `LookupEvents` (or the console) supports targeted investigations: deletions, approval changes, failed calls, per-user activity.
✅ **EventBridge rules on CloudTrail events** turn the audit log into real-time controls (deletion alerts, unapproved-model detection, audit databases).
✅ **State-based compliance checks** (approved models on endpoints, required tags) complement event-based auditing.
✅ For >90 days: **trails → S3 → Athena** (or CloudTrail Lake); harden with log validation, KMS, GuardDuty, Config, Security Hub.

## Workshop Section Complete 🎉

You've completed the Lab 5 monitoring trilogy:

| Lab | Layer | Question it answers |
|---|---|---|
| **5A** CloudWatch Metrics | Operational health | *Is it fast, available, and right-sized?* |
| **5B** CloudWatch Logs | Debugging | *Why did it fail?* |
| **5C** CloudTrail | Governance | *Who did what, and was it allowed?* |

Together these close the ML lifecycle loop: models that are not just built and deployed with governance (Labs 1–4), but **operated** with full observability and auditability.
